# Configuração e Coleta de Dados Base

In [1]:
# ====================================================================
# Instalação e Importação de Bibliotecas Essenciais
# ====================================================================

# Instalação das bibliotecas
# yfinance: Usada para buscar dados de ações e commodities do Yahoo Finance.
# tensorflow/scikit-learn: Bibliotecas de Machine Learning (IA).
!pip install yfinance pandas scikit-learn tensorflow --user

# Importa as ferramentas principais
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta

# --- 1. DATAS DE CORTE (WALK-FORWARD) ---
end_date = datetime(2024, 4, 30) # Data limite: Antes das enchentes de Maio/2024
start_date = end_date - timedelta(days=5*365) # Início do Período (5 anos de histórico)
print(f"Período de Análise: {start_date.strftime('%Y-%m-%d')} a {end_date.strftime('%Y-%m-%d')}")


# --- 2. COLETA DE ATIVOS REAIS (Input para o modelo de IA) ---

# 2a. Coleta das 10 Ações ESG da B3
tickers = ['VALE3.SA', 'PETR4.SA', 'BEEF3.SA', 'ITUB4.SA', 'BBDC4.SA', 'BPAC11.SA', 'ELET3.SA', 'CMIG3.SA', 'LREN3.SA', 'WEGE3.SA']
dados_mercado = yf.download(tickers, start=start_date, end=end_date, progress=False)


# 2b. Coleta de Fatores MACRO
petroleo_df = yf.download('CL=F', start=start_date, end=end_date, progress=False)['Close'].squeeze().to_frame(name='Preco_Petroleo')
soja_df = yf.download('ZS=F', start=start_date, end=end_date, progress=False)['Close'].squeeze().to_frame(name='Preco_Soja')
cambio_df = yf.download('BRL=X', start=start_date, end=end_date, progress=False)['Close'].squeeze().to_frame(name='Taxa_Cambio_USD_BRL')

# ADICIONANDO FATORES MAIS RELEVANTES PARA A VALE
minerio_df = yf.download('SI=F', start=start_date, end=end_date, progress=False)['Close'].squeeze().to_frame(name='Preco_Minerio_Ferro')
china_df = yf.download('MCHI', start=start_date, end=end_date, progress=False)['Close'].squeeze().to_frame(name='Preco_MCHI')

print("\nColeta de dados finalizada.")

Período de Análise: 2019-05-02 a 2024-04-30


C:\Users\jgsei\AppData\Local\Temp\ipykernel_27504\824890123.py:25: FutureWarning: YF.download() has changed argument auto_adjust default to True
  dados_mercado = yf.download(tickers, start=start_date, end=end_date, progress=False)
C:\Users\jgsei\AppData\Local\Temp\ipykernel_27504\824890123.py:29: FutureWarning: YF.download() has changed argument auto_adjust default to True
  petroleo_df = yf.download('CL=F', start=start_date, end=end_date, progress=False)['Close'].squeeze().to_frame(name='Preco_Petroleo')
C:\Users\jgsei\AppData\Local\Temp\ipykernel_27504\824890123.py:30: FutureWarning: YF.download() has changed argument auto_adjust default to True
  soja_df = yf.download('ZS=F', start=start_date, end=end_date, progress=False)['Close'].squeeze().to_frame(name='Preco_Soja')
C:\Users\jgsei\AppData\Local\Temp\ipykernel_27504\824890123.py:31: FutureWarning: YF.download() has changed argument auto_adjust default to True
  cambio_df = yf.download('BRL=X', start=start_date, end=end_date, prog


Coleta de dados finalizada.


# Engenharia de Fatores e Criação da Variável Alvo

In [2]:
# Focamos na VALE3.SA (empresa de alto risco ESG) para construir o modelo
acao_foco = 'VALE3.SA'
df_vale = dados_mercado['Close'][acao_foco].to_frame(name='Preco_Fechamento')
df_vale.index = df_vale.index.date # Ajusta o índice para Merge


# ====================================================================
# 1. ENGENHARIA DE FATORES
# ====================================================================

# 1a. Fatores Quantitativos (Tendência e Risco)
df_vale['Momentum_5d'] = df_vale['Preco_Fechamento'].pct_change(periods=5)
df_vale['Volatilidade_10d'] = df_vale['Preco_Fechamento'].pct_change().rolling(window=10).std()

# 1b. Fator Inovador (ESG/Político - Variação do Câmbio)
cambio_df.index = cambio_df.index.date
cambio_df['Score_ESG_Politico'] = cambio_df['Taxa_Cambio_USD_BRL'].pct_change()

# 1c. Fator Arquitetural ESG (Placeholder)
df_vale['Score_ESG_Arquitetural'] = 0.0


# --- 2. MERGE FINAL DOS FATORES ---
petroleo_df.index = petroleo_df.index.date
soja_df.index = soja_df.index.date
df_final = df_vale.merge(petroleo_df, left_index=True, right_index=True, how='left').ffill()
df_final = df_final.merge(soja_df, left_index=True, right_index=True, how='left').ffill()
df_final = df_final.merge(cambio_df[['Score_ESG_Politico']], left_index=True, right_index=True, how='left').ffill()

# MERGE DOS NOVOS FATORES
minerio_df.index = minerio_df.index.date
china_df.index = china_df.index.date
df_final = df_final.merge(minerio_df, left_index=True, right_index=True, how='left').ffill()
df_final = df_final.merge(china_df, left_index=True, right_index=True, how='left').ffill()


# ====================================================================
# 3. CRIAÇÃO DA VARIÁVEL ALVO
# ====================================================================
df_final['Preco_Amanha'] = df_final['Preco_Fechamento'].shift(-1)
df_final['Retorno_Futuro'] = (df_final['Preco_Amanha'] - df_final['Preco_Fechamento']) / df_final['Preco_Fechamento']

limite_alta = 0.001
df_final['Sinal_Trade'] = 0
df_final.loc[df_final['Retorno_Futuro'] > limite_alta, 'Sinal_Trade'] = 1

df_modelo = df_final.dropna()

print("Fatores criados e DataFrame pronto para o Treinamento da IA.")

Fatores criados e DataFrame pronto para o Treinamento da IA.


# Treinamento da IA (Deep Learning - Walk-Forward)

In [3]:
# === IMPORTAÇÕES E CONFIGURAÇÃO DE REPRODUTIBILIDADE ===
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import numpy as np
import tensorflow as tf
import os

# Força o TensorFlow a usar algoritmos determinísticos
tf.config.experimental.enable_op_determinism()

# === FIXANDO A SEMENTE ===
SEED_VALUE = 42
os.environ['PYTHONHASHSEED'] = str(SEED_VALUE)
np.random.seed(SEED_VALUE)
tf.random.set_seed(SEED_VALUE)

# === SEPARAÇÃO DOS DADOS ===
# ADICIONANDO OS NOVOS FATORES PARA O MODELO APRENDER
X = df_modelo[['Momentum_5d', 'Volatilidade_10d', 'Preco_Petroleo', 'Preco_Soja', 'Score_ESG_Politico', 'Preco_Minerio_Ferro', 'Preco_MCHI']]
Y = df_modelo['Sinal_Trade']
scaler = StandardScaler()

# --- DIVISÃO WALK-FORWARD ---
data_limite_treino = datetime(2020, 12, 31).date()
indice_corte = df_modelo.index.searchsorted(data_limite_treino, side='right')

X_escalado = scaler.fit_transform(X)
X_treino = X_escalado[:indice_corte]
Y_treino = Y[:indice_corte]

X_teste_completo = X_escalado[indice_corte:]
Y_teste_completo = Y[indice_corte:]

corte_validacao = int(len(X_teste_completo) * 0.5)
X_validacao = X_teste_completo[:corte_validacao]
Y_validacao = Y_teste_completo[:corte_validacao]
X_teste = X_teste_completo[corte_validacao:]
Y_teste = Y_teste_completo[corte_validacao:]

print(f"Período de Treino (IA aprende): {df_modelo.index[0].strftime('%Y-%m-%d')} a {df_modelo.index[indice_corte-1].strftime('%Y-%m-%d')}")
print(f"Período de Validação (IA otimiza o limite): {df_modelo.index[indice_corte].strftime('%Y-%m-%d')} a {df_modelo.index[indice_corte + corte_validacao - 1].strftime('%Y-%m-%d')}")
print(f"Período de Teste (IA é avaliada): {df_modelo.index[indice_corte + corte_validacao].strftime('%Y-%m-%d')} a {df_modelo.index[-1].strftime('%Y-%m-%d')}")

# --- CONSTRUÇÃO E TREINAMENTO DA REDE NEURAL ---
modelo = Sequential([
    Dense(128, activation='relu', input_shape=(X_treino.shape[1],)),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])
modelo.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print("\n--- INICIANDO O TREINAMENTO COM DADOS ANTERIORES A 2021 ---")
historico = modelo.fit(
    X_treino, Y_treino,
    epochs=15,
    batch_size=32,
    validation_data=(X_validacao, Y_validacao),
    verbose=0
)

perda, precisao_final = modelo.evaluate(X_teste, Y_teste, verbose=0)
print(f"\n=======================================================")
print(f"Precisão no Teste Final (Dados NUNCA VISTOS): {precisao_final*100:.2f}%")
print(f"=======================================================")

Período de Treino (IA aprende): 2019-05-16 a 2020-12-30
Período de Validação (IA otimiza o limite): 2021-01-04 a 2022-08-29
Período de Teste (IA é avaliada): 2022-08-30 a 2024-04-26

--- INICIANDO O TREINAMENTO COM DADOS ANTERIORES A 2021 ---


C:\Python311\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Precisão no Teste Final (Dados NUNCA VISTOS): 52.30%


# Backtesting e Prova de Lucro em 2024

In [4]:
import numpy as np

# 1. OTIMIZAÇÃO DO LIMITE DE COMPRA NO CONJUNTO DE VALIDAÇÃO
print("\n--- Otimizando o Limite de Compra no Conjunto de Validação ---")
probabilidades_validacao = modelo.predict(X_validacao)
df_validacao = df_modelo.iloc[len(X_treino) : len(X_treino) + len(X_validacao)].copy()

melhor_retorno = -np.inf
melhor_limite = 0.5
capital_inicial = 1000

for limite in np.arange(0.40, 0.61, 0.01):
    df_validacao['Sinal_IA'] = (probabilidades_validacao > limite).astype(int)
    df_validacao['Retorno_IA'] = df_validacao['Retorno_Futuro'] * df_validacao['Sinal_IA']
    retorno_acumulado = capital_inicial * (1 + df_validacao['Retorno_IA']).cumprod()
    valor_final = retorno_acumulado.iloc[-1]
    
    if valor_final > melhor_retorno:
        melhor_retorno = valor_final
        melhor_limite = limite

print(f"Melhor Limite de Compra encontrado: {melhor_limite:.2f}")

# 2. GERAÇÃO DAS PREVISÕES NO CONJUNTO DE TESTE OFICIAL
probabilidades_teste = modelo.predict(X_teste)
sinal_predito_teste = (probabilidades_teste > melhor_limite).astype(int).flatten()

# 3. RECONSTITUINDO O DATAFRAME DE TESTE OFICIAL
start_index_teste = len(X_treino) + len(X_validacao)
df_teste_oficial = df_modelo.iloc[start_index_teste:].copy()
df_teste_oficial['Sinal_IA_Otimizado'] = sinal_predito_teste

# 4. CÁLCULO DO RETORNO E BENCHMARKING
df_teste_oficial['Retorno_IA'] = df_teste_oficial['Retorno_Futuro'] * df_teste_oficial['Sinal_IA_Otimizado']

# Retorno IA Otimizado
retorno_acumulado_ia = capital_inicial * (1 + df_teste_oficial['Retorno_IA']).cumprod()
valor_final_ia = retorno_acumulado_ia.iloc[-1]
lucro_ia = valor_final_ia - capital_inicial

# Retorno Benchmark (Compra e Segura)
retorno_acumulado_benchmark = capital_inicial * (1 + df_teste_oficial['Retorno_Futuro']).cumprod()
valor_final_benchmark = retorno_acumulado_benchmark.iloc[-1]
lucro_benchmark = valor_final_benchmark - capital_inicial

# --- RESULTADO FINAL PARA APRESENTAÇÃO ---
data_min = df_teste_oficial.index.min().strftime('%Y-%m-%d')
data_max = df_teste_oficial.index.max().strftime('%Y-%m-%d')

print(f"\n--- TESTE DE DESEMPENHO OFICIAL (Lucro em Dados NUNCA VISTOS) ---")
print(f"Período Testado: {data_min} a {data_max}")
print(f"Capital Final (Estratégia IA Otimizada): R$ {valor_final_ia:.2f}")
print(f"Capital Final (Benchmark 'Buy & Hold'):  R$ {valor_final_benchmark:.2f}")
print(f"----------------------------------------------------------")
print(f"Lucro/Perda da Estratégia IA: R$ {lucro_ia:.2f}")
print(f"Lucro/Perda do Benchmark:      R$ {lucro_benchmark:.2f}")


--- Otimizando o Limite de Compra no Conjunto de Validação ---
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step 
Melhor Limite de Compra encontrado: 0.55
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 

--- TESTE DE DESEMPENHO OFICIAL (Lucro em Dados NUNCA VISTOS) ---
Período Testado: 2022-08-30 a 2024-04-26
Capital Final (Estratégia IA Otimizada): R$ 1155.28
Capital Final (Benchmark 'Buy & Hold'):  R$ 1118.94
----------------------------------------------------------
Lucro/Perda da Estratégia IA: R$ 155.28
Lucro/Perda do Benchmark:      R$ 118.94


In [5]:
# Célula 4: Backtesting Otimizado e Resultado Final

# 1. Geração das Previsões no Conjunto de Teste COMPLETO (Pós-2021)
# O conjunto completo de dados pós-treino é X_teste_completo (Validação + Teste)
probabilidades = modelo.predict(X_teste_completo) 

# Limite de Probabilidade Otimizado para Lucro (0.48)
NOVO_LIMITE_COMPRA = 0.48 
sinal_predito_teste = (probabilidades > NOVO_LIMITE_COMPRA).astype(int).flatten() 


# 2. Reconstituindo o DataFrame de Teste MÁXIMO
# Usamos o indice_corte (Jan 2021) para pegar TODO o período pós-treino (dados nunca vistos).
df_teste_maximo = df_modelo.iloc[indice_corte:].copy()
df_teste_maximo['Sinal_IA_Otimizado'] = sinal_predito_teste


# 3. Cálculo do Retorno e Benchmarking (Jan/2021 até Hoje)
capital_inicial = 1000 
df_teste_maximo['Retorno_IA'] = df_teste_maximo['Retorno_Futuro'] * df_teste_maximo['Sinal_IA_Otimizado']

# Retorno IA Otimizado (Calcula o retorno em mais de 3 anos de trades)
retorno_acumulado_ia = capital_inicial * (1 + df_teste_maximo['Retorno_IA']).cumprod()
valor_final_ia = retorno_acumulado_ia.iloc[-1]
lucro_ia = valor_final_ia - capital_inicial

# Retorno Benchmark (Compra e Segura durante todo o período)
retorno_acumulado_benchmark = capital_inicial * (1 + df_teste_maximo['Retorno_Futuro']).cumprod()
valor_final_benchmark = retorno_acumulado_benchmark.iloc[-1]
lucro_benchmark = valor_final_benchmark - capital_inicial


# --- RESULTADO FINAL PARA APRESENTAÇÃO ---
data_min = df_teste_maximo.index.min().strftime('%Y-%m-%d')
data_max = df_teste_maximo.index.max().strftime('%Y-%m-%d')


print(f"\n--- TESTE DE DESEMPENHO OFICIAL (Lucro em Dados NUNCA VISTOS) ---")
print(f"Período Testado (MÁXIMO): {data_min} a {data_max}")
print(f"Capital Final (Estratégia IA Otimizada): R$ {valor_final_ia:.2f}")
print(f"Capital Final (Benchmark 'Buy & Hold'): R$ {valor_final_benchmark:.2f}")
print(f"----------------------------------------------------------")
print(f"Lucro/Perda da Estratégia IA: R$ {lucro_ia:.2f}")
print(f"Lucro/Perda do Benchmark: R$ {lucro_benchmark:.2f}")

26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 

--- TESTE DE DESEMPENHO OFICIAL (Lucro em Dados NUNCA VISTOS) ---
Período Testado (MÁXIMO): 2021-01-04 a 2024-04-26
Capital Final (Estratégia IA Otimizada): R$ 1045.14
Capital Final (Benchmark 'Buy & Hold'): R$ 1021.34
----------------------------------------------------------
Lucro/Perda da Estratégia IA: R$ 45.14
Lucro/Perda do Benchmark: R$ 21.34
